# Agilent BenchCel 4R quickstart

The Agilent BenchCel 4R is a four-stacker microplate handler. PyLabRobot controls its arm, stacker mechanisms, grippers, teachpoints, status queries, and labware configuration over Ethernet.

| Property | Value |
|---|---|
| Communication | Ethernet TCP |
| Default port | `7612` |
| Frame | command byte + 16-bit little-endian payload length + payload |
| Configuration | Four stackers and one taught transfer position |
| Labware | ANSI/SLAS-format plates with device-specific gripper settings |
| Verified with | BenchCel 4R, firmware `3.2.20.0` |

```{note}
This driver was verified against a live BenchCel 4R and VWorks packet captures. The protocol is reverse-engineered rather than vendor-published, so verify motions at low risk on your own hardware before unattended operation.
```

```{device-card} agilent-benchcel-4r
```

## How it communicates

The BenchCel accepts binary command frames over TCP port 7612. Successful commands normally complete with a `0x69` acknowledgement; device faults arrive as `0x02` frames containing an ASCII error message. Motion acknowledgements are returned after the motion finishes.

Only one effective control client should own the connection. Close VWorks before connecting from PyLabRobot, and stop PyLabRobot before reconnecting VWorks.

## Physical setup

Before connecting:

1. Keep the pendant's robot-disable button and the emergency stop accessible.
2. Clear the arm and stacker workspace of people, tools, and loose labware.
3. Turn on compressed air and confirm the stacker racks are correctly installed and supported.
4. Connect the control computer to the BenchCel Ethernet network and confirm its IP address.
5. Teach and verify the transfer position in VWorks. Record its numeric teachpoint ID.
6. Confirm the physical plate matches the configured labware dimensions and gripper offsets.
7. Close VWorks so it does not compete for the control session.

```{warning}
An incorrect teachpoint can send the arm toward a home-like position. Never copy a teachpoint ID from another instrument without verifying it on this BenchCel.
```

## Create the device

Set the instrument IP address and the transfer teachpoint ID verified on your BenchCel. The `0x1E` value below was observed on one installation and must not be assumed to be universal.

In [ ]:
from pylabrobot.agilent import BenchCel4R

BENCHCEL_IP = "192.168.0.10"  # Replace with this instrument's address.
TRANSFER_TEACHPOINT_ID = 0x1E  # Replace with the ID verified in VWorks.

benchcel = BenchCel4R(
  name="benchcel",
  host=BENCHCEL_IP,
  loading_tray_teachpoint_id=TRANSFER_TEACHPOINT_ID,
)

## Connect

Open the TCP connection after VWorks is closed.

In [ ]:
await benchcel.setup()

## Check arm status

Read the current theta, X, Z, and robot-gripper positions before commanding motion.

In [ ]:
arm_pose = await benchcel.request_arm_pose()
arm_pose

## Check the stacker sensors

Query all four stackers. `plate_presence` is an analog-like value; `plate_present()` applies the driver's default threshold.

In [ ]:
sensor_statuses = await benchcel.request_all_stacker_sensor_statuses()
[
  {
    "stacker": status.stacker,
    "air_pressure": status.air_pressure,
    "plate_presence": status.plate_presence,
    "plate_present": status.plate_present(),
  }
  for status in sensor_statuses.values()
]

## Model the labware

Use the PLR resource factory that matches the physical plate. This example uses the Corning 3603 definition; replace it with your exact model. If plates nest, also set the plate resource’s `stacking_z_height` to the measured vertical pitch so the resource stack geometry remains accurate.

In [ ]:
from pylabrobot.resources import cor_96_wellplate_360uL_Fb


def example_plate(name: str):
  return cor_96_wellplate_360uL_Fb(name)

## Configure the BenchCel labware profile

Push geometry only after checking every value against the physical plate and the VWorks setup. Invalid or mismatched gripper settings can drop or damage labware.

In [ ]:
await benchcel.set_labware(example_plate("benchcel_labware_profile"))
benchcel.labware_settings

## Track the plates in a stacker

Populate each `ResourceStack` in bottom-to-top order. The last assigned plate is the accessible top plate.

In [ ]:
stack_1 = benchcel.stacks[0]
for index in range(3):
  stack_1.assign_child_resource(example_plate(f"stack_1_plate_{index + 1}"))

[plate.name for plate in stack_1.children]

## Home the motors

Home only with the full workspace clear. The controller normally drops its TCP session during homing; `home()` reconnects and waits until the device responds again.

In [ ]:
await benchcel.home()

## Downstack a plate

Move the accessible top plate from stacker 1 to the taught transfer position.

In [ ]:
plate = await benchcel.downstack(stack_1)

## Upstack a plate

Return that plate from the taught transfer position to stacker 1. Confirm the plate is still present at the transfer position before running this command.

In [ ]:
await benchcel.upstack(stack_1)

## Stacker and teaching diagnostics

The API also exposes absolute moves with `move_to_stacker()` and `move_to_teachpoint()`, robot-gripper positioning with `open_robot_gripper()` and `close_robot_gripper()`, stacker-clamp positioning with `open_stacker_clamps()` and `close_stacker_clamps()`, and teachpoint writes with `set_teachpoint()`. These are service and teaching operations rather than a hello-world workflow. In particular, opening a stacker's pneumatic clamps can release the entire plate stack. Use these methods only with the rack supported and the service procedure for your installation in hand.

## Disconnect

Close the TCP connection before opening VWorks or ending the protocol.

In [ ]:
await benchcel.stop()